In [1]:
# import pandas as pd
# import numpy as np
# import os
# from sklearn.preprocessing import MinMaxScaler
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Dense
# from tensorflow.keras.optimizers import Adam

# # ۱. خواندن فایل اصلی
# file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_generator_bearings_deviation_monitoring\deviation_monitoring\dsas_g11_generator_bearings_deviation_monitoring_output5.xlsx'

# # لیست فیچرها و تارگت‌ها
# all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
# # لیست سنسورهایی که تارگت هستند
# target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
# # ۲. بارگذاری و پیش‌پردازش داده‌ها
# df = pd.read_excel(file_path)
# df['date'] = pd.to_datetime(df['date'])
# df = df.sort_values(by='date')

# # حذف ردیف‌های خالی در سنسورهای حیاتی
# df_clean = df.dropna(subset=all_features).copy()
# raw_data = df_clean[all_features].values

# # ۳. نرمال‌سازی داده‌ها (Min-Max Scaling)
# scaler = MinMaxScaler()
# scaled_data = scaler.fit_transform(raw_data)

# # ۴. ساخت معماری Autoencoder (بازسازی چندمتغیره)
# input_dim = len(all_features)
# input_layer = Input(shape=(input_dim,))

# # Encoder: استخراج ویژگی‌های سیستماتیک
# encoded = Dense(16, activation='relu')(input_layer)
# latent = Dense(8, activation='relu')(encoded)

# # Decoder: بازسازی سیگنال‌ها
# decoded = Dense(16, activation='relu')(latent)
# output_layer = Dense(input_dim, activation='sigmoid')(decoded)

# autoencoder = Model(inputs=input_layer, outputs=output_layer)
# autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')

# # ۵. آموزش مدل روی تمام داده‌ها (یادگیری رفتار نرمال سیستم)
# print("🚀 مرحله ۱: یادگیری رفتار سیستماتیک سنسورها...")
# autoencoder.fit(scaled_data, scaled_data, epochs=150, batch_size=32, shuffle=True, verbose=0)

# # ۶. تولید باقیمانده‌ها (Residuals) و شاخص انحراف
# reconstructed = autoencoder.predict(scaled_data)
# # محاسبه MSE برای هر لحظه (شاخص خام ناهنجاری)
# mse_errors = np.mean(np.power(scaled_data - reconstructed, 2), axis=1)
# df_clean['Raw_Anomaly_Index'] = mse_errors

# # ۷. ارزیابی پیشرفته باقیمانده‌ها (Advanced Residual Evaluation)

# # الف) فیلتر EWMA برای شناسایی "روند" و حذف نویز لحظه‌ای
# # alpha=0.1 باعث می‌شود تغییرات تدریجی خرابی بهتر دیده شود
# df_clean['Smooth_Deviation_Index'] = df_clean['Raw_Anomaly_Index'].ewm(alpha=0.1).mean()

# # ب) تحلیل آماری برای تعیین آستانه‌های پویا (Dynamic Thresholds)
# # استفاده از صدک‌های داده‌های تاریخی برای قضاوت
# p95_threshold = df_clean['Smooth_Deviation_Index'].quantile(0.95)
# p99_threshold = df_clean['Smooth_Deviation_Index'].quantile(0.99)

# # ج) منطق تصمیم‌گیری (قضاوت نهایی)
# def final_judgment(val):
#     if val > p99_threshold:
#         return "Critical (Red) - Systematic Failure"
#     elif val > p95_threshold:
#         return "Warning (Yellow) - Operational Drift"
#     else:
#         return "Normal (Green)"

# df_clean['Final_Status'] = df_clean['Smooth_Deviation_Index'].apply(final_judgment)


# # ۸. فیلتر کردن خروجی برای "یک ماه آخر"
# last_date = df_clean['date'].max()
# start_of_last_month = last_date - pd.Timedelta(days=30)
# df_output = df_clean[df_clean['date'] >= start_of_last_month].copy()

# # ۹. ذخیره خروجی نهایی
# os.makedirs(os.path.dirname(output_filename), exist_ok=True)
# df_output.to_excel(output_filename, index=False)

# print("-" * 50)
# print(f"✅ تحلیل هوشمند با موفقیت به پایان رسید.")
# print(f"📈 آستانه هشدار (زرد): {p95_threshold:.6f}")
# print(f"🚫 آستانه بحرانی (قرمز): {p99_threshold:.6f}")
# print(f"📊 بازه زمانی خروجی: {df_output['date'].min()} تا {df_output['date'].max()}")
# print(f"📂 فایل خروجی در مسیر زیر ذخیره شد:\n{output_filename}")

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

def run_autoencoder_anomaly_detection():
    """اجرای تحلیل ناهنجاری با Autoencoder برای ژنراتور و ذخیره خروجی"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # ۱. خواندن فایل اصلی
    file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
    output_filename = r'outputs\G11\dsas_g11_generator_bearings_deviation_monitoring\deviation_monitoring\dsas_g11_generator_bearings_deviation_monitoring_output5.xlsx'
    
    # ایجاد پوشه خروجی
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)

    # لیست فیچرها و تارگت‌ها
    all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                    'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    
    target_sensors = all_features  # همه سنسورها تارگت هستند

    # ۲. بارگذاری و پیش‌پردازش داده‌ها
    try:
        df = pd.read_excel(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values(by='date')
        print(f"✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: {len(df):,}")
        print(f"📅 بازه زمانی: {df['date'].min()} تا {df['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return None

    # حذف ردیف‌های خالی در سنسورهای حیاتی
    df_clean = df.dropna(subset=all_features).copy()
    raw_data = df_clean[all_features].values
    print(f"📊 تعداد رکوردهای بدون داده خالی: {len(df_clean):,}")

    # ۳. نرمال‌سازی داده‌ها (Min-Max Scaling)
    print("🔄 مرحله 1: نرمال‌سازی داده‌ها...")
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(raw_data)
    print(f"   ✅ نرمال‌سازی با {len(all_features)} سنسور انجام شد")

    # ۴. ساخت معماری Autoencoder
    print("🔄 مرحله 2: ساخت و آموزش مدل Autoencoder...")
    
    input_dim = len(all_features)
    input_layer = Input(shape=(input_dim,))

    # Encoder: استخراج ویژگی‌های سیستماتیک
    encoded = Dense(16, activation='relu')(input_layer)
    latent = Dense(8, activation='relu')(encoded)

    # Decoder: بازسازی سیگنال‌ها
    decoded = Dense(16, activation='relu')(latent)
    output_layer = Dense(input_dim, activation='sigmoid')(decoded)

    autoencoder = Model(inputs=input_layer, outputs=output_layer)
    autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')

    # ۵. آموزش مدل روی تمام داده‌ها
    print("   🚀 مرحله ۱: یادگیری رفتار سیستماتیک سنسورها...")
    autoencoder.fit(scaled_data, scaled_data, epochs=150, batch_size=32, shuffle=True, verbose=0)
    print("   ✅ آموزش مدل با موفقیت انجام شد")

    # ۶. تولید باقیمانده‌ها (Residuals) و شاخص انحراف
    print("🔄 مرحله 3: محاسبه شاخص‌های انحراف...")
    
    reconstructed = autoencoder.predict(scaled_data, verbose=0)
    # محاسبه MSE برای هر لحظه (شاخص خام ناهنجاری)
    mse_errors = np.mean(np.power(scaled_data - reconstructed, 2), axis=1)
    df_clean['Raw_Anomaly_Index'] = mse_errors
    print(f"   ✅ شاخص خام ناهنجاری محاسبه شد")

    # ۷. ارزیابی پیشرفته باقیمانده‌ها (Advanced Residual Evaluation)
    print("🔄 مرحله 4: ارزیابی پیشرفته باقیمانده‌ها...")

    # الف) فیلتر EWMA برای شناسایی "روند" و حذف نویز لحظه‌ای
    df_clean['Smooth_Deviation_Index'] = df_clean['Raw_Anomaly_Index'].ewm(alpha=0.1).mean()
    print(f"   ✅ فیلتر EWMA اعمال شد")

    # ب) تحلیل آماری برای تعیین آستانه‌های پویا (Dynamic Thresholds)
    p95_threshold = df_clean['Smooth_Deviation_Index'].quantile(0.95)
    p99_threshold = df_clean['Smooth_Deviation_Index'].quantile(0.99)
    
    print(f"   📈 آستانه هشدار (زرد): {p95_threshold:.6f}")
    print(f"   🚫 آستانه بحرانی (قرمز): {p99_threshold:.6f}")

    # ج) منطق تصمیم‌گیری (قضاوت نهایی)
    def final_judgment(val):
        if val > p99_threshold:
            return "Critical (Red) - Systematic Failure"
        elif val > p95_threshold:
            return "Warning (Yellow) - Operational Drift"
        else:
            return "Normal (Green)"

    df_clean['Final_Status'] = df_clean['Smooth_Deviation_Index'].apply(final_judgment)
    
    # نمایش توزیع وضعیت‌ها
    status_counts = df_clean['Final_Status'].value_counts()
    print(f"\n📊 توزیع وضعیت‌ها در کل داده‌ها:")
    for status, count in status_counts.items():
        print(f"   {status}: {count:,} ({count/len(df_clean)*100:.2f}%)")

    # ۸. فیلتر کردن خروجی برای "یک ماه آخر"
    print("🔄 مرحله 5: فیلتر کردن داده‌های یک ماه اخیر...")
    
    last_date = df_clean['date'].max()
    start_of_last_month = last_date - pd.Timedelta(days=30)
    df_output = df_clean[df_clean['date'] >= start_of_last_month].copy()
    
    print(f"   📅 بازه خروجی: از {df_output['date'].min()} تا {df_output['date'].max()}")
    print(f"   تعداد رکوردهای ماه اخیر: {len(df_output):,}")
    
    # نمایش توزیع وضعیت‌ها در ماه اخیر
    status_counts_last_month = df_output['Final_Status'].value_counts()
    print(f"\n📊 توزیع وضعیت‌ها در ماه اخیر:")
    for status, count in status_counts_last_month.items():
        print(f"   {status}: {count:,} ({count/len(df_output)*100:.2f}%)")

    # ۹. ذخیره خروجی نهایی
    print("💾 مرحله 6: ذخیره خروجی...")
    
    try:
        df_output.to_excel(output_filename, index=False)
        print(f"✅ تحلیل هوشمند با موفقیت به پایان رسید.")
        print(f"📂 فایل خروجی در مسیر زیر ذخیره شد:\n{output_filename}")
        print(f"📊 تعداد رکوردهای نهایی: {len(df_output):,}")
        print(f"📋 تعداد ستون‌ها: {len(df_output.columns)}")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return df_output

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش ناهنجاری ژنراتور با Autoencoder")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["23:00", "23:01", "23:02"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_autoencoder_anomaly_detection()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه پایش ناهنجاری ژنراتور با Autoencoder")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")


🚀 شروع برنامه پایش ناهنجاری ژنراتور با Autoencoder
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش ناهنجاری ژنراتور با Autoencoder
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-01 23:00:03
🔄 شروع تحلیل در 2026-07-01 23:00:03
✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: 11,915
📅 بازه زمانی: 2021-03-16 05:33:48 تا 2026-05-31 20:30:17
📊 تعداد رکوردهای بدون داده خالی: 11,915
🔄 مرحله 1: نرمال‌سازی داده‌ها...
   ✅ نرمال‌سازی با 9 سنسور انجام شد
🔄 مرحله 2: ساخت و آموزش مدل Autoencoder...
   🚀 مرحله ۱: یادگیری رفتار سیستماتیک سنسورها...
   ✅ آموزش مدل با موفقیت انجام شد
🔄 مرحله 3: محاسبه شاخص‌های انحراف...
   ✅ شاخص خام ناهنجاری محاسبه شد
🔄 مرحله 4: ارزیابی پیشرفته باقیمانده‌ها...
   ✅ فیلتر EWMA اعمال شد
   📈 آستانه هشدار (زرد): 0.001259
   🚫 آستانه بحرانی (قرمز): 0.003088

📊 توزیع وضعیت‌ها در کل داده‌ها:
   Normal (Green): 11,319 (95.00%)
   Warning (Yellow) - Operational Drift: 476 (3.99%)
  